# DDoS Detection and Botmaster Attribution — CTU-13 Scenario 10

This notebook builds and evaluates the full pipeline: data loading,
feature extraction, DDoS detection, and three-signal (timing/rhythm/
structure) botmaster attribution, fused into one ranked suspect list.

Cleaned version: keeps only the final, working version of each step.
Dead-end attempts we tried and discarded (e.g. classical CUSUM, TCP-flag
based filtering) are noted briefly in markdown where they explain a design
decision, but their code is not repeated here — see the project report for
the full debugging narrative.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import correlate
import matplotlib.pyplot as plt

In [ ]:
# CTU-13 Scenario 10 (rbot malware, 10 infected hosts, UDP/ICMP flood attack,
# IRC-based C2). Ground truth (infected IPs, attack timeline) taken from the
# scenario's README.
columns = ['StartTime','Dur','Proto','SrcAddr','Sport','Dir','DstAddr','Dport',
           'State','sTos','dTos','TotPkts','TotBytes','SrcBytes','Label']

df = pd.read_csv('capture20110818.binetflow', names=columns, header=0, low_memory=False)
df['StartTime'] = pd.to_datetime(df['StartTime'])
len(df)

## 2. Data Exploration

Sanity-check the data against the README's ground truth before building
anything on top of it.

In [ ]:
# How rare is attack traffic relative to total network traffic?
df['Label'].value_counts()

In [ ]:
# Confirm the botnet-labelled source IPs match the README's 10 infected hosts exactly.
botnet_df = df[df['Label'].str.contains('Botnet', na=False)]
botnet_df['SrcAddr'].value_counts()

In [ ]:
# Note: 'Botnet' labels are host-based (any flow to/from a known-infected IP),
# not behaviour-based — the earliest labelled flows are ordinary DNS/HTTP
# traffic, not attack traffic. This is discussed as a limitation in the report.
botnet_df.sort_values('StartTime').head(3)

## 3. Feature Extraction

In [ ]:
# Bucket flows into 5-second windows and sum packets per window — the
# volumetric feature the detector will monitor.
df = df.set_index('StartTime')
packet_rate = df['TotPkts'].resample('5s').sum()
df = df.reset_index()

**Note on a distortion we found and did not fully correct:** a flow's
entire packet count is attributed to the single 5-second window in which it
*started*, even if the flow ran for many minutes (NetFlow records only a
start time, not per-second packet timing). This concentrates long flows
(e.g. the attack floods) into one window rather than spreading them across
their true duration. An alternative "active flow count" feature was tested
as a possible fix but found insensitive to this scenario's volumetric floods
(see report Section 4.1) and was not used further.

## 4. Detection

A classical CUSUM detector was tried first but found unusable on this data:
its cumulative memory never resets on this heavily right-skewed traffic
distribution, so it becomes permanently "stuck" in an alarm state after the
first anomaly (see report Section 4.1 for the full diagnosis). It was
replaced with a memoryless local-outlier detector below.

In [ ]:
def local_outlier_detect(series, rolling_median, rolling_iqr, k=10):
    """Flag a window as anomalous if it exceeds its own recent local
    median by k times the local IQR. Memoryless (no running total carried
    forward), which avoids the CUSUM 'stuck alarm' failure mode observed
    on this dataset.
    """
    alarms = []
    in_alarm = False
    for i, x in enumerate(series):
        is_outlier = x > rolling_median.iloc[i] + k * rolling_iqr.iloc[i]
        if is_outlier and not in_alarm:
            alarms.append(i)
            in_alarm = True
        elif not is_outlier:
            in_alarm = False
    return alarms

In [ ]:
# Cap extreme outlier windows (99th percentile) so a single huge background
# flow can't dominate the local baseline calculation.
cap_value = packet_rate.quantile(0.99)
capped_packet_rate = packet_rate.clip(upper=cap_value)

# Short (2-minute) rolling window: a 1-hour window was found too slow to
# adapt to the network's natural daytime traffic increase, causing false
# accumulation unrelated to any attack.
window_size = 24  # 24 x 5s = 2 minutes
rolling_median = capped_packet_rate.rolling(window=window_size, min_periods=5).median()
rolling_iqr = (capped_packet_rate.rolling(window=window_size, min_periods=5).quantile(0.75)
               - capped_packet_rate.rolling(window=window_size, min_periods=5).quantile(0.25))

alarms = local_outlier_detect(capped_packet_rate.values, rolling_median, rolling_iqr, k=10)
alarm_times = [capped_packet_rate.index[i] for i in alarms]
len(alarm_times)

In [ ]:
# Ground-truth attack episodes, manually transcribed from the scenario README's
# operator timeline (attack start/stop commands).
known_attacks = [
    ('2011-08-18 11:52:54', '2011-08-18 12:07:30'),
    ('2011-08-18 12:08:43', '2011-08-18 12:11:54'),
    ('2011-08-18 12:12:20', '2011-08-18 12:14:53'),
    ('2011-08-18 13:56:52', '2011-08-18 14:05:48'),
    ('2011-08-18 14:06:12', '2011-08-18 14:07:25'),
    ('2011-08-18 14:09:04', '2011-08-18 14:09:29'),
    ('2011-08-18 14:10:08', '2011-08-18 14:10:43'),
    ('2011-08-18 14:14:47', '2011-08-18 14:15:59'),
    ('2011-08-18 14:30:02', '2011-08-18 14:30:38'),
    ('2011-08-18 14:41:27', '2011-08-18 14:42:07'),
    ('2011-08-18 14:43:27', '2011-08-18 14:44:04'),
    ('2011-08-18 15:00:46', '2011-08-18 15:03:42'),
    ('2011-08-18 15:16:38', '2011-08-18 15:25:51'),
    ('2011-08-18 15:32:04', '2011-08-18 15:35:38'),
]
known_attacks = [(pd.Timestamp(s), pd.Timestamp(e)) for s, e in known_attacks]
len(known_attacks)

In [ ]:
def is_near_attack(alarm_time, known_attacks, tolerance_seconds=30):
    tol = pd.Timedelta(seconds=tolerance_seconds)
    return any((start - tol) <= alarm_time <= (end + tol) for start, end in known_attacks)

true_positives = [t for t in alarm_times if is_near_attack(t, known_attacks)]
false_positives = [t for t in alarm_times if not is_near_attack(t, known_attacks)]

precision = len(true_positives) / (len(true_positives) + len(false_positives))
caught_attacks = sum(
    any(is_near_attack(t, [(start, end)]) for t in alarm_times)
    for start, end in known_attacks
)
recall = caught_attacks / len(known_attacks)
f1 = 2 * precision * recall / (precision + recall)

print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

### 4.1 Confusion Matrix and ROC/AUC

**Note:** the precision/recall/F1 above are computed at *episode* granularity (caught vs missed attack episodes, alarms near vs away from an episode) rather than per-window, because that is what the assignment's detection metric needs. For a proper confusion matrix and ROC curve we need per-*window* ground truth: was each 5-second window part of a known attack or not, and was it alarmed or not, across a range of thresholds `k`. This was not computed in the original clean notebook — added here so it can be produced on a full rerun against the raw capture.

In [ ]:
# Per-window ground truth: label every 5s window as attack (1) or benign (0).
# (This was missing from the notebook in the previous version of this cell --
# it was only added to a local test script and never written back here.)
window_is_attack = pd.Series(0, index=capped_packet_rate.index)
for _start, _end in known_attacks:
    window_is_attack.loc[_start:_end] = 1

def in_alarm_state(k):
    """Reconstruct the full per-window in-alarm boolean state for threshold k,
    using exactly the same is_outlier test and hysteresis rule as
    local_outlier_detect (entry and reset both governed by the same k).
    An earlier version of this cell used a hardcoded k=10 for the reset
    condition regardless of the k being tested, which silently corrupted
    the ROC sweep below (fixed after running against the real capture file
    made the bug visible: FPR/TPR barely moved across k).
    """
    series = capped_packet_rate.values
    y_pred = np.zeros(len(series), dtype=int)
    alarm_flag = False
    for i, x in enumerate(series):
        is_outlier = x > rolling_median.iloc[i] + k * rolling_iqr.iloc[i]
        if is_outlier:
            alarm_flag = True
        elif not is_outlier:
            alarm_flag = False
        y_pred[i] = 1 if alarm_flag else 0
    return y_pred

def confusion_counts_at_k(k, n_windows):
    y_true = window_is_attack.values
    y_pred = in_alarm_state(k)
    tp = int(((y_pred==1) & (y_true==1)).sum())
    fp = int(((y_pred==1) & (y_true==0)).sum())
    tn = int(((y_pred==0) & (y_true==0)).sum())
    fn = int(((y_pred==0) & (y_true==1)).sum())
    return tp, fp, tn, fn

n_windows = len(capped_packet_rate)
tp, fp, tn, fn = confusion_counts_at_k(10, n_windows)
print('Confusion matrix (per-window, k=10):')
print(f'  TP={tp}  FP={fp}')
print(f'  FN={fn}  TN={tn}')
print(f'  window-level precision={tp/(tp+fp):.3f}  recall={tp/(tp+fn):.3f}')


In [ ]:
# ROC / AUC by sweeping the outlier threshold k across a wide enough range
# to reach both extremes (flag almost everything / almost nothing), so the
# curve spans close to the full [0,1] FPR range and AUC is comparable to the
# usual 0.5-is-chance interpretation.
from sklearn.metrics import auc

k_values = np.concatenate([np.arange(-3, 1, 0.25), np.arange(1, 30, 1), np.arange(30, 200, 10)])
tpr_list, fpr_list = [], []
for k in k_values:
    tp, fp, tn, fn = confusion_counts_at_k(k, n_windows)
    tpr_list.append(tp / (tp + fn) if (tp+fn) > 0 else 0)
    fpr_list.append(fp / (fp + tn) if (fp+tn) > 0 else 0)

# sort by FPR and append the trivial (0,0) / (1,1) endpoints
pts = sorted(zip(fpr_list, tpr_list))
fpr_sorted = [0.0] + [p[0] for p in pts] + [1.0]
tpr_sorted = [0.0] + [p[1] for p in pts] + [1.0]

roc_auc = auc(fpr_sorted, tpr_sorted)
plt.figure(figsize=(5,5))
plt.plot(fpr_sorted, tpr_sorted, marker='o', markersize=3, label=f'AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1],'--', color='grey', label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Detection ROC curve (per-window, threshold k swept)')
plt.legend()
plt.savefig('detection_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC = {roc_auc:.3f}')


**To run this on your machine:** this needs the actual `capture20110818.binetflow` file loaded in Section 1 and the detector cells above already executed (so `capped_packet_rate`, `rolling_median`, `rolling_iqr`, and `alarms` exist). Running it will produce a real confusion matrix and an AUC value — paste those numbers into report Section 5.1 rather than the placeholder framing above. `scikit-learn` may need `pip install scikit-learn` if not already available.

## 5. Attribution — Candidate Discovery

Build the timeline used for attribution (everything before the first
attack), and find the set of hosts that had genuine, sustained pre-attack
contact with the known bot IPs.

In [ ]:
# Ground-truth infected hosts, from the scenario README.
bot_ips = ['147.32.84.165','147.32.84.191','147.32.84.192','147.32.84.193',
           '147.32.84.204','147.32.84.205','147.32.84.206','147.32.84.207',
           '147.32.84.208','147.32.84.209']

first_attack_start = pd.Timestamp('2011-08-18 11:52:54')
pre_attack = df[df['StartTime'] < first_attack_start]

# Attack-burst indicator series (1 during any known attack window), used by
# the timing signal below.
attack_burst = pd.Series(0, index=packet_rate.index)
for start, end in known_attacks:
    attack_burst.loc[start:end] = 1

In [ ]:
# TotPkts > 10 filters out two kinds of noise we found by inspection:
# (a) port-scan traffic (many distinct ports, 2 packets each, all 'Attempt'
#     labelled) and (b) rejected/retried TCP handshakes, which Argus can
#     report with a long 'Dur' despite almost no data actually moving.
# A stricter filter requiring a graceful TCP close (FIN flag) was tried and
# rejected: the real IRC C2 sessions never close gracefully (they stay open
# listening for commands), so that filter excluded genuine C2 traffic too.
clean_talk = pre_attack[
    (pre_attack['TotPkts'] > 10) &
    ((pre_attack['SrcAddr'].isin(bot_ips)) | (pre_attack['DstAddr'].isin(bot_ips)))
].copy()

clean_talk['duration_minutes'] = clean_talk['Dur'] / 60
clean_ranked = clean_talk.groupby(
    clean_talk.apply(lambda r: r['DstAddr'] if r['SrcAddr'] in bot_ips else r['SrcAddr'], axis=1)
)['duration_minutes'].sum().sort_values(ascending=False)
clean_ranked

In [ ]:
# Independent corroboration: check whether the unfamiliar top-ranked hosts
# also use IRC (port 6667) + identd (port 113), the same signature as the
# manually-confirmed C2 servers.
for ip in clean_ranked.index:
    ports = pre_attack[
        (pre_attack['SrcAddr'] == ip) | (pre_attack['DstAddr'] == ip)
    ]['Dport'].value_counts().head(3)
    print(ip)
    print(ports)
    print()

## 6. Timing Clue (Lead-Lag Cross-Correlation)

In [ ]:
def timing_score(candidate_ip, df, bot_ips, attack_burst, packet_rate_index, max_lag_windows=180):
    """Cross-correlate a candidate's pre-attack activity against the
    attack-burst series. Only positive lags up to max_lag_windows (here,
    180 windows = 15 minutes) are considered plausible 'candidate leads
    attack' timing — unconstrained search was found to pick up spurious
    long-range/negative-lag alignments for hosts with no real relationship
    to the attack. The raw correlation is normalised by the candidate's
    total active-window count: without this, a high-traffic but innocent
    host (the network's DNS server) outscored genuine C2 hosts purely by
    having many more opportunities for incidental overlap.
    """
    flows = df[
        ((df['SrcAddr']==candidate_ip) & (df['DstAddr'].isin(bot_ips))) |
        ((df['DstAddr']==candidate_ip) & (df['SrcAddr'].isin(bot_ips)))
    ]
    activity = pd.Series(0, index=packet_rate_index)
    for t in flows['StartTime']:
        matching = packet_rate_index[packet_rate_index <= t]
        if len(matching) == 0:
            continue
        activity.loc[matching[-1]] = 1

    total_active = activity.sum()
    if total_active == 0:
        return {'lag_windows': None, 'raw_corr': 0, 'norm_score': 0, 'active_windows': 0}

    corr = correlate(attack_burst.values, activity.values, mode='full')
    lags = np.arange(-len(activity)+1, len(attack_burst))
    valid = (lags >= 0) & (lags <= max_lag_windows)
    if not valid.any():
        return {'lag_windows': None, 'raw_corr': 0, 'norm_score': 0, 'active_windows': total_active}

    corr_valid = corr[valid]
    lags_valid = lags[valid]
    best_idx = np.argmax(corr_valid)
    return {
        'lag_windows': lags_valid[best_idx],
        'raw_corr': corr_valid[best_idx],
        'norm_score': corr_valid[best_idx] / total_active,
        'active_windows': total_active
    }

In [ ]:
# Full candidate set: confirmed C2 hosts plus two innocent hosts used as
# adversarial test cases (the network's DNS server, and an ordinary
# browsing host with zero bot contact).
candidates_to_test = list(clean_ranked.index) + ['147.32.80.9', '147.32.84.59']

timing_results = []
for ip in candidates_to_test:
    r = timing_score(ip, df, bot_ips, attack_burst, packet_rate.index, max_lag_windows=180)
    r['host'] = ip
    timing_results.append(r)

timing_results_df = pd.DataFrame(timing_results).sort_values('norm_score', ascending=False)
timing_results_df

## 7. Rhythm Clue (Connection Regularity)

In [ ]:
def beaconing_cv(candidate_ip, df, bot_ips, min_connections=3):
    """Coefficient of variation (CV = std/mean) of inter-connection gap
    times. Lower CV = more regular/machine-like timing. Note: classic tight
    'beacon every N seconds' behaviour (CV near 0) was not observed for this
    botnet, since it uses persistent IRC connections rather than periodic
    polling — CV here instead captures general timing regularity, which
    still discriminated C2 from innocent hosts in testing (see report).
    """
    flows = df[
        ((df['SrcAddr']==candidate_ip) & (df['DstAddr'].isin(bot_ips))) |
        ((df['DstAddr']==candidate_ip) & (df['SrcAddr'].isin(bot_ips)))
    ].sort_values('StartTime')
    timestamps = flows['StartTime']
    if len(timestamps) < min_connections:
        return {'n_connections': len(timestamps), 'cv': None}
    gaps = timestamps.diff().dropna().dt.total_seconds()
    cv = gaps.std() / gaps.mean() if gaps.mean() > 0 else None
    return {'n_connections': len(timestamps), 'cv': cv}

def rhythm_score(cv, max_cv=8):
    """max_cv=8 chosen as a normalisation ceiling based on the observed
    range of candidate CVs in this scenario (innocent DNS host reached
    CV ~= 7.2; confirmed C2 hosts fell in 1.0-1.9).
    """
    if cv is None:
        return 0
    return max(0, 1 - cv / max_cv)

In [ ]:
rhythm_results = []
for ip in candidates_to_test:
    r = beaconing_cv(ip, df, bot_ips)
    r['host'] = ip
    r['rhythm_score'] = rhythm_score(r['cv']) if r['cv'] is not None else 0
    rhythm_results.append(r)

rhythm_df = pd.DataFrame(rhythm_results).sort_values('rhythm_score', ascending=False)
rhythm_df

## 8. Structure Clue (Targeted Fan-Out)

**Known limitation (see report Section 6.2):** every confirmed C2 host here
has only 1-2 total pre-attack contacts (they are external servers with no
other reason to appear on this network), which mechanically forces their
fan-out ratio toward 1.0 regardless of genuine targeting intent. The signal
still discriminates correctly here, but for a partly confounded reason —
see the report for full discussion.

In [ ]:
def fanout_score(candidate_ip, bot_ips, pre_attack):
    """Fraction of a candidate's total pre-attack contacts that are known
    bot IPs, rather than raw count — so being 'busy' alone isn't scored as
    suspicious, only being disproportionately connected to eventual
    attackers.
    """
    candidate_flows = pre_attack[
        (pre_attack['SrcAddr']==candidate_ip) | (pre_attack['DstAddr']==candidate_ip)
    ]
    contacts = pd.concat([
        candidate_flows.loc[candidate_flows['SrcAddr']==candidate_ip, 'DstAddr'],
        candidate_flows.loc[candidate_flows['DstAddr']==candidate_ip, 'SrcAddr']
    ]).unique()
    contacts = set(contacts) - {candidate_ip}
    if len(contacts) == 0:
        return {'total_contacts': 0, 'bot_contacts': 0, 'fanout_score': 0}
    bot_overlap = contacts & set(bot_ips)
    return {
        'total_contacts': len(contacts),
        'bot_contacts': len(bot_overlap),
        'fanout_score': len(bot_overlap) / len(contacts)
    }

In [ ]:
structure_results = []
for ip in candidates_to_test:
    f = fanout_score(ip, bot_ips, pre_attack)
    f['host'] = ip
    structure_results.append(f)

structure_df = pd.DataFrame(structure_results).sort_values('fanout_score', ascending=False)
structure_df

## 9. Fusion — Composite Ranking

In [ ]:
fusion_rows = []
for ip in candidates_to_test:
    t = timing_score(ip, df, bot_ips, attack_burst, packet_rate.index, max_lag_windows=180)
    r = beaconing_cv(ip, df, bot_ips)
    f = fanout_score(ip, bot_ips, pre_attack)
    fusion_rows.append({
        'host': ip,
        'timing': t['norm_score'],
        'rhythm': rhythm_score(r['cv']) if r['cv'] is not None else 0,
        'structure': f['fanout_score'],
        'structure_n': f['total_contacts']
    })

fusion_df = pd.DataFrame(fusion_rows)

# Equal weighting (1/3 each) — a simple, explainable starting choice.
weights = {'timing': 1/3, 'rhythm': 1/3, 'structure': 1/3}
fusion_df['composite'] = (
    weights['timing'] * fusion_df['timing'] +
    weights['rhythm'] * fusion_df['rhythm'] +
    weights['structure'] * fusion_df['structure']
)

fusion_ranked = fusion_df.sort_values('composite', ascending=False).reset_index(drop=True)
fusion_ranked

## 10. Evaluation (Top-k Accuracy, MRR)

**Note:** with 8 true C2 hosts among 10 total candidates, standard Top-k
accuracy is mathematically bounded well below 1.0 for small k even for a
perfect ranking (at most 1 true host can occupy rank 1, at most 3 can
occupy the top 3, etc.). Top-8 accuracy is the most informative statistic
for this configuration — see report Section 5.2 for full discussion.

In [ ]:
true_c2_hosts = {'82.96.64.4', '89.16.176.16', '93.152.160.101', '213.179.58.83',
                  '213.232.93.3', '86.65.39.15', '38.229.70.20', '78.40.125.4'}

ranked_hosts = list(fusion_ranked['host'])
ranks = {host: ranked_hosts.index(host) + 1 for host in true_c2_hosts}

def top_k_accuracy(ranks_dict, k):
    return sum(1 for r in ranks_dict.values() if r <= k) / len(ranks_dict)

for k in [1, 3, 5, 8]:
    print(f"Top-{k} accuracy: {top_k_accuracy(ranks, k):.2f}")

mrr = np.mean([1/r for r in ranks.values()])
print(f"MRR: {mrr:.3f}")

## 11. Ablation Study

Full per-signal ablation: each of the three signals alone, all three pairs, and the full fusion, evaluated with Top-k accuracy, MRR, and a margin measure (gap between the weakest true C2 host and the strongest innocent host's composite score). Top-k/MRR are included for completeness but, as found below, cannot distinguish between variants here because any subset of signals already achieves perfect rank-separation (a ceiling effect of this dataset, not a property of the metric in general) — margin is the informative statistic in this scenario.

The single-signal variants below (timing-only, rhythm-only, structure-only) also serve as internal baselines: each corresponds to a simpler existing approach from the literature reviewed in Section 2 (timing-only ~ lead-lag correlation approaches such as BotSniffer's temporal correlation; rhythm-only ~ beaconing/periodicity-based detection; structure-only ~ fan-out/graph-based approaches). Comparing these to full fusion tests whether the fusion actually improves on any one signal used alone.

In [ ]:
innocent_hosts = {'147.32.80.9', '147.32.84.59'}

variants = {
    'timing_only':      ['timing'],
    'rhythm_only':      ['rhythm'],
    'structure_only':   ['structure'],
    'timing+rhythm':    ['timing', 'rhythm'],
    'timing+structure': ['timing', 'structure'],
    'rhythm+structure': ['rhythm', 'structure'],
    'full_fusion':      ['timing', 'rhythm', 'structure'],
}

def top_k_accuracy(ranks_dict, k):
    return sum(1 for r in ranks_dict.values() if r <= k) / len(ranks_dict)

ablation_rows = []
for name, cols in variants.items():
    fusion_df[f'score_{name}'] = fusion_df[cols].mean(axis=1)
    scores = fusion_df.set_index('host')[f'score_{name}']

    ranked_hosts_variant = scores.sort_values(ascending=False).index.tolist()
    ranks_variant = {h: ranked_hosts_variant.index(h) + 1 for h in true_c2_hosts}
    mrr_variant = np.mean([1/r for r in ranks_variant.values()])

    weakest_true = scores[list(true_c2_hosts)].min()
    strongest_innocent = scores[list(innocent_hosts)].max()

    row = {'variant': name,
           'MRR': round(mrr_variant, 3),
           'margin': round(weakest_true - strongest_innocent, 3)}
    for k in [1, 3, 5, 8]:
        row[f'top{k}'] = round(top_k_accuracy(ranks_variant, k), 3)
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows)
ablation_df


**Result:** Top-k accuracy and MRR are identical across every variant (0.34 MRR, Top-8 = 1.00) — with 8 of 10 candidates being true positives, any two-of-three (or even one) signal already achieves perfect Top-8 separation in this scenario, so rank-based metrics are saturated and cannot show which signals matter. The margin measure does distinguish them:

| Variant | Margin (Scenario 10) |
|---|---|
| Rhythm only | 0.660 (best single signal) |
| Rhythm + Structure | 0.575 |
| Structure only | 0.450 |
| Full fusion (all three) | 0.473 |
| Timing + Rhythm | 0.425 |
| Timing + Structure | 0.360 |
| Timing only | 0.100 (weakest single signal) |

**This complicates the novelty claim and the report says so directly:** full fusion does *not* achieve the largest margin in Scenario 10 — rhythm alone, and rhythm+structure without timing, both separate C2 hosts from innocent hosts more cleanly. Removing rhythm causes the single largest margin drop (0.473 → 0.360, a fall of 0.113), confirming it is the most load-bearing signal here. Removing timing *increases* the margin (0.473 → 0.575), consistent with the report's finding that the current timing normalisation may add noise for a subset of hosts rather than a genuine detection improvement.

Scenario 11 tells a different story: timing alone achieves the *largest* margin (1.00), because rhythm is completely uninformative there (every candidate has too few pre-attack connections for a CV to be computed, so rhythm contributes a flat 0 for all hosts and only dilutes the average). Full fusion's margin (0.50) sits *below* timing-only for this reason.

**Honest conclusion for the report:** the fused composite is not shown to strictly outperform every single signal on margin in either scenario. Its practical value in this evaluation is *robustness* — it degrades gracefully rather than collapsing when one signal is locally uninformative (Scenario 11's rhythm) or noisy (Scenario 10's timing), rather than superior peak separation over the best single signal. This is a more modest, more defensible novelty claim than 'fusion improves attribution', and should replace it in Section 3/7 of the report.